# 03 — Logistic regression baseline

This notebook fits the first cancellation model under the preparation and evaluation rules fixed in notebooks 01 and 02. It uses only training outcomes known by validation start, produces one booking-time cancellation score for each validation booking, and evaluates that score in the weekly risk-only queue.

Logistic regression is the simple baseline for booking-time risk ranking because it is quick to fit, produces probabilities, and is easier to explain than a more complex model. There is no extensive tuning here, and holdout is not evaluated.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display
PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "hotel_bookings.csv").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "modeling"))

import evaluation_policy as evaluation
from models import (
    CATEGORICAL_FEATURES, NUMERIC_FEATURES, make_logistic_pipeline,
)
from prepare_data import (
    RAW_DATA_PATH, complete_180_rows,
    model_features, prepare_data,
)

pd.set_option("display.max_columns", 60)

## 1. Recreate the approved training and validation populations

The nominal training period contains 60 booking weeks. A row is used for fitting only when its arrival is before validation begins, so its cancellation or no-show outcome was already known at that cutoff. This gives the 43,264-row fitting population defined in notebook 02.

Validation contains the following 13 booking weeks. Holdout rows are deliberately not selected or scored in this notebook.

In [2]:
raw = pd.read_csv(RAW_DATA_PATH)
prepared = prepare_data(raw)
modeling_population = complete_180_rows(prepared)

validation_population = modeling_population.loc[
    modeling_population["booking_period_split"].eq("validation")
].copy()
validation_start = validation_population["booking_week_start"].min()

nominal_train = modeling_population.loc[
    modeling_population["booking_period_split"].eq("train")
].copy()
model_fit_population = nominal_train.loc[
    nominal_train["arrival_date"].lt(validation_start)
].copy()

X_fit = model_features(model_fit_population)
y_fit = model_fit_population["is_canceled"].copy()
X_validation = model_features(validation_population)
y_validation = validation_population["is_canceled"].copy()

population_summary = pd.DataFrame(
    {
        "rows": [len(model_fit_population), len(validation_population)],
        "booking weeks": [
            model_fit_population["booking_week_start"].nunique(),
            validation_population["booking_week_start"].nunique(),
        ],
        "cancellation rate": [y_fit.mean(), y_validation.mean()],
    },
    index=["model fitting", "validation"],
)
population_summary.style.format({"cancellation rate": "{:.2%}"})

,rows,booking weeks,cancellation rate
model fitting,43264,60,29.15%
validation,11050,13,31.76%


## 2. Fit one simple pipeline and check regularization strength

Categorical values are one-hot encoded, with unseen validation categories ignored rather than causing an error. Numeric values are standardized so their different units do not control the regularized model. The prepared features contain no missing values, so another imputation layer is unnecessary.

The reusable pipeline definition lives in `modeling/models.py`; this notebook retains the fitting, sensitivity comparison, and explanation. The ordinary L2-regularized logistic regression is used without class weights. Cancellations are common enough that class balancing is not needed, and artificial class weights would make the predicted probabilities harder to interpret. All preprocessing is fitted only on `X_fit` inside the pipeline.

One small sensitivity check compares `C = 0.1`, `1`, and `10`. Smaller `C` means stronger regularization. Each candidate is fitted on the same rows and evaluated with the same validation queue. The predeclared choice is the highest mean weekly precision@150; an exact tie is resolved by lower Brier score and then smaller `C`. This uses the dedicated validation period rather than random cross-validation, and holdout remains untouched.

In [3]:
display(
    pd.Series(
        {
            "numeric features": len(NUMERIC_FEATURES),
            "categorical features": len(CATEGORICAL_FEATURES),
            "total features before encoding": (
                len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES)
            ),
        },
        name="count",
    ).to_frame()
)
print(
    "Missing feature values — fitting:",
    int(X_fit.isna().sum().sum()),
    "| validation:",
    int(X_validation.isna().sum().sum()),
)

,count
numeric features,16
categorical features,11
total features before encoding,27


Missing feature values — fitting: 0 | validation: 0


In [4]:
candidate_models = {}
candidate_probabilities = {}
sensitivity_rows = []

for c_value in [0.1, 1.0, 10.0]:
    candidate_pipeline = make_logistic_pipeline(c_value)
    candidate_pipeline.fit(X_fit, y_fit)
    p_cancel = candidate_pipeline.predict_proba(X_validation)[:, 1]

    candidate_scored = validation_population.copy()
    candidate_scored["p_cancel"] = p_cancel
    _, _, candidate_queue_summary, _ = (
        evaluation.simulate_weekly_policy(
            candidate_scored,
            "validation",
            policy="risk_only",
        )
    )
    candidate_metrics = evaluation.model_metric_summary(
        y_validation, p_cancel
    )

    candidate_models[c_value] = candidate_pipeline
    candidate_probabilities[c_value] = p_cancel
    sensitivity_rows.append(
        {
            "C": c_value,
            "mean weekly precision@150": candidate_queue_summary[
                "mean weekly queue precision at capacity"
            ],
            "average precision": candidate_metrics["average_precision"],
            "Brier score": candidate_metrics["brier_score"],
        }
    )

sensitivity_results = pd.DataFrame(sensitivity_rows)
selection_order = sensitivity_results.sort_values(
    ["mean weekly precision@150", "Brier score", "C"],
    ascending=[False, True, True],
)
selected_c = float(selection_order.iloc[0]["C"])
logistic_pipeline = candidate_models[selected_c]

validation_scored = validation_population.copy()
validation_scored["p_cancel"] = candidate_probabilities[selected_c]

display(
    sensitivity_results.style.format(
        {
            "C": "{:g}",
            "mean weekly precision@150": "{:.2%}",
            "average precision": "{:.4f}",
            "Brier score": "{:.4f}",
        }
    )
)
print("Selected C:", f"{selected_c:g}")
display(
    validation_scored["p_cancel"]
    .agg(["min", "mean", "median", "max"])
    .rename("selected predicted cancellation probability")
    .to_frame()
)

,C,mean weekly precision@150,average precision,Brier score
0,0.1,57.95%,0.7339,0.1623
1,1,58.05%,0.7313,0.1633
2,10,58.00%,0.7287,0.1639


Selected C: 1


,selected predicted cancellation probability
min,4.178394e-59
mean,2.119707e-01
median,9.357881e-02
max,9.994424e-01


## 3. Check the validation probabilities

Average precision measures how well higher-risk bookings are ranked, while the Brier score checks probability error and is lower when probabilities are closer to the outcomes. ROC AUC is included only as a familiar supporting measure. The constant baseline assigns every validation row the fitting-period cancellation rate.

These checks describe the model generally. The weekly queue metric in the next section remains the primary comparison measure.

In [5]:
constant_probability = pd.Series(y_fit.mean(), index=y_validation.index)
probability_metrics = pd.DataFrame(
    {
        "logistic regression": evaluation.model_metric_summary(
            y_validation, validation_scored["p_cancel"]
        ),
        "constant training-rate baseline": evaluation.model_metric_summary(
            y_validation, constant_probability
        ),
    }
).T
probability_metrics.round(4)

,average_precision,roc_auc,brier_score
logistic regression,0.7313,0.8344,0.1633
constant training-rate baseline,0.3176,0.5000,0.2174


### Calibration by probability group

For a well-calibrated model, the mean predicted probability and observed cancellation rate should be reasonably close within each group. This table is a diagnostic, not another tuning target.

In [6]:
validation_scored["probability_group"] = pd.qcut(
    validation_scored["p_cancel"], q=10, duplicates="drop"
)
calibration_table = (
    validation_scored.groupby("probability_group", observed=True)
    .agg(
        bookings=("source_row_id", "size"),
        mean_predicted_probability=("p_cancel", "mean"),
        observed_cancellation_rate=("is_canceled", "mean"),
    )
)
calibration_table.style.format(
    {
        "mean_predicted_probability": "{:.2%}",
        "observed_cancellation_rate": "{:.2%}",
    }
)

,bookings,mean_predicted_probability,observed_cancellation_rate
probability_group,,,
"(-0.001, 0.00556]",1105,0.06%,1.54%
"(0.00556, 0.0211]",1105,1.39%,3.98%
"(0.0211, 0.0379]",1105,2.90%,8.69%
"(0.0379, 0.0612]",1144,4.96%,22.20%
"(0.0612, 0.0936]",1066,7.70%,19.61%
"(0.0936, 0.148]",1105,11.76%,27.78%
"(0.148, 0.228]",1105,18.65%,37.10%
"(0.228, 0.355]",1112,28.25%,46.40%
"(0.355, 0.649]",1098,48.18%,64.03%


## 4. Evaluate the risk-only weekly queue

This is the model-comparison policy fixed in notebook 02. Each Monday, the active and uncalled validation queue is ranked only by the immutable `p_cancel`, with up to 150 calls. Booking value and urgency are intentionally excluded until a model has been selected.

The main result is mean weekly precision@150. A repeated random ranking uses the same queue rules and capacity, showing what the operation might reach without model-based ordering.

In [7]:
(
    logistic_calls,
    logistic_weekly,
    logistic_queue_summary,
    logistic_final_queue,
) = evaluation.simulate_weekly_policy(
    validation_scored,
    "validation",
    policy="risk_only",
)

display(
    logistic_weekly[
        [
            "call_date",
            "eligible_queue",
            "calls",
            "cancellations_reached",
            "precision_at_capacity",
        ]
    ].style.format({"precision_at_capacity": "{:.2%}"})
)

queue_fields = [
    "total calls",
    "cancellations reached",
    "mean weekly queue precision at capacity",
    "weekly precision IQR",
    "mean capacity utilization",
    "bookings still waiting after final decision",
]
logistic_queue_summary.loc[queue_fields].to_frame("logistic regression")

,call_date,eligible_queue,calls,cancellations_reached,precision_at_capacity
0,2016-09-05 00:00:00,542,150,84,56.00%
1,2016-09-12 00:00:00,840,150,71,47.33%
2,2016-09-19 00:00:00,1177,150,95,63.33%
3,2016-09-26 00:00:00,1722,150,79,52.67%
4,2016-10-03 00:00:00,1936,150,64,42.67%
5,2016-10-10 00:00:00,1991,150,79,52.67%
6,2016-10-17 00:00:00,2285,150,78,52.00%
7,2016-10-24 00:00:00,2677,150,99,66.00%
8,2016-10-31 00:00:00,2689,150,92,61.33%
9,2016-11-07 00:00:00,2647,150,79,52.67%


,logistic regression
total calls,1950.000000
cancellations reached,1132.000000
mean weekly queue precision at capacity,0.580513
weekly precision IQR,0.133333
mean capacity utilization,1.000000
bookings still waiting after final decision,2583.000000


In [8]:
random_results = evaluation.repeated_random_baseline(
    validation_scored,
    "validation",
    repeats=100,
)
primary_metric = "mean weekly queue precision at capacity"

queue_comparison = pd.Series(
    {
        "logistic regression": logistic_queue_summary[primary_metric],
        "random mean": random_results[primary_metric].mean(),
        "random 5th percentile": random_results[primary_metric].quantile(0.05),
        "random 95th percentile": random_results[primary_metric].quantile(0.95),
    },
    name="mean weekly precision@150",
)
queue_comparison.map(lambda value: f"{value:.2%}").to_frame()

,mean weekly precision@150
logistic regression,58.05%
random mean,24.84%
random 5th percentile,23.47%
random 95th percentile,25.90%


## 5. Inspect the strongest model associations

Positive coefficients are associated with higher predicted cancellation risk and negative coefficients with lower predicted risk, after accounting for the other inputs. Numeric features were standardized; categorical coefficients describe encoded category levels.

These are predictive associations, not evidence that changing a feature would change cancellation behavior. Correlated inputs can also share or shift coefficient weight, so the table is a compact model check rather than a causal explanation.

In [9]:
encoded_feature_names = logistic_pipeline.named_steps[
    "preprocessing"
].get_feature_names_out()
coefficients = pd.Series(
    logistic_pipeline.named_steps["model"].coef_[0],
    index=encoded_feature_names,
    name="coefficient",
)

strongest_higher = coefficients.nlargest(8).reset_index()
strongest_higher["association"] = "higher predicted risk"
strongest_lower = coefficients.nsmallest(8).reset_index()
strongest_lower["association"] = "lower predicted risk"
coefficient_summary = pd.concat(
    [strongest_higher, strongest_lower], ignore_index=True
).rename(columns={"index": "encoded feature"})
coefficient_summary[["association", "encoded feature", "coefficient"]].round(3)

,association,encoded feature,coefficient
0,higher predicted risk,categorical__agent_281,2.892
1,higher predicted risk,categorical__agent_208,2.732
2,higher predicted risk,categorical__deposit_type_Non Refund,2.504
3,higher predicted risk,categorical__country_NGA,1.978
4,higher predicted risk,categorical__agent_315,1.975
5,higher predicted risk,categorical__agent_134,1.949
6,higher predicted risk,categorical__agent_240,1.919
7,higher predicted risk,categorical__country_HKG,1.842
8,lower predicted risk,numeric__required_car_parking_spaces,-4.354
9,lower predicted risk,categorical__agent_298,-3.643


## 6. What this baseline shows and what comes next

The regularization sensitivity is nearly flat: `C = 0.1`, `1`, and `10` produce mean weekly precision@150 of 57.95%, 58.05%, and 58.00%. Under the selection rule stated before running the comparison, `C = 1` is selected. It reaches only one more cancellation than `C = 10` and two more than `C = 0.1` across 1,950 calls, so this is not evidence of a meaningful tuning improvement; it simply supports retaining the ordinary default. `C = 0.1` has slightly better average precision and Brier score, which is reported rather than hidden, but those were supporting measures rather than the primary choice rule.

The model provides useful ranking: average precision is 0.731 compared with the 0.318 validation cancellation rate, and the risk-only queue reaches 1,132 cancellations in 1,950 calls. Its mean weekly precision@150 is 58.05%, compared with a 24.84% mean across 100 random queue runs.

The probabilities are not perfectly calibrated. Their validation mean is 21.20% while 31.76% of validation bookings canceled, and most probability groups underpredict the observed rate. This does not erase the strong ranking result, but `p_cancel` should not yet be described as a precise probability or expected-loss estimate. No recalibration is fitted on validation here because that would reuse the same outcomes used to judge the model.

Several of the largest coefficients belong to agent, company, or country levels with limited data, so they should not be treated as stable business explanations. Logistic regression remains the first candidate, not yet the chosen model. The next notebook should fit CatBoost on exactly the same rows and features and evaluate the same validation queue. Only after that comparison should a model and business policy be selected. Holdout remains untouched until both are frozen.